In [ ]:
Laila Said: laila_204265

Leen Ghoneem: leen_161577

Adnan Al-As'ad: adnan_189704

In [ ]:
# basic stuff
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# sklearn
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (mean_squared_error, mean_absolute_error, r2_score,
                             accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report, silhouette_score)
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering

np.random.seed(42)
print("imports done")

# colab might be blocked behind a 403 Forbidden error, this cell is for mitigating that
import sklearn.datasets
from sklearn.utils import Bunch

def fake_fetch(as_frame=True):
    url = "https://raw.githubusercontent.com/ageron/handson-ml2/master/datasets/housing/housing.csv"
    d = pd.read_csv(url)
    d['total_bedrooms'] = d['total_bedrooms'].fillna(d['total_bedrooms'].median())

    data = pd.DataFrame({
        'MedInc': d['median_income'],
        'HouseAge': d['housing_median_age'],
        'AveRooms': d['total_rooms'] / d['households'],
        'AveBedrms': d['total_bedrooms'] / d['households'],
        'Population': d['population'],
        'AveOccup': d['population'] / d['households'],
        'Latitude': d['latitude'],
        'Longitude': d['longitude']
    })

    target = d['median_house_value'] / 100000.0

    frame = data.copy()
    frame['MedHouseVal'] = target

    return Bunch(data=data.values, target=target.values, frame=frame, feature_names=list(data.columns))

sklearn.datasets.fetch_california_housing = fake_fetch
from sklearn.datasets import fetch_california_housing

In [ ]:
# Load the dataset with fetch_california_housing(as_frame=True)
housing = fetch_california_housing(as_frame=True)
df = housing.frame.copy()

# rename target to match the assignment
df.rename(columns={'MedHouseVal': 'MedHouseValue'}, inplace=True)

print(df.shape)
df.info()
df.isnull().sum()
df.head()
df.describe()

# split features and target
X = df.drop(columns=['MedHouseValue'])
y = df['MedHouseValue']

# 80/20 split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(X_train.shape, X_test.shape)

<div dir="rtl">
<b>ملخص الجزء الأول.</b> تحتوي البيانات على 20,640 صفاً و8 خصائص، وهي: MedInc و HouseAge و AveRooms و AveBedrms و Population و AveOccup و Latitude و Longitude. المتغير الهدف هو MedHouseValue (قيمة المنزل بوحدة 100,000 دولار). تم تقسيم البيانات بنسبة 80% للتدريب و20% للاختبار.
</div>

In [ ]:
# baseline linear regression
lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)
y_pred = lin_reg.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("RMSE:", round(rmse, 4))
print("MAE:", round(mae, 4))
print("R2:", round(r2, 4))

In [ ]:
# ridge and lasso, need scaling first
scal = StandardScaler()

X_train_2 = scal.fit_transform(X_train)
X_test_2 = scal.transform(X_test)

ridge = Ridge(alpha=1.0)
ridge.fit(X_train_2, y_train)
y_pred_ridge = ridge.predict(X_test_2)

lasso = Lasso(alpha=0.1)
lasso.fit(X_train_2, y_train)
y_pred_lasso = lasso.predict(X_test_2)

print("Ridge:", round(np.sqrt(mean_squared_error(y_test, y_pred_ridge)), 4),
      round(r2_score(y_test, y_pred_ridge), 4))
print("Lasso:", round(np.sqrt(mean_squared_error(y_test, y_pred_lasso)), 4),
      round(r2_score(y_test, y_pred_lasso), 4))

In [ ]:
# polynomial features for MedInc and HouseAge only
poly_cols = ['MedInc', 'HouseAge']

pre = ColumnTransformer([('poly', PolynomialFeatures(degree=2, include_bias=False), poly_cols)],
                        remainder='passthrough')

poly_model = Pipeline([('pre', pre), ('scaler', StandardScaler()), ('lin', LinearRegression())])
poly_model.fit(X_train, y_train)
y_pred_poly = poly_model.predict(X_test)

print("Poly RMSE:", round(np.sqrt(mean_squared_error(y_test, y_pred_poly)), 4))
print("Poly R2:", round(r2_score(y_test, y_pred_poly), 4))

In [ ]:
# which features matter most
coefs = pd.DataFrame({'Feature': X_train.columns, 'Coef': lin_reg.coef_})
coefs['AbsCoef'] = coefs['Coef'].abs()
coefs = coefs.sort_values('AbsCoef', ascending=False)
coefs = coefs.reset_index(drop=True)
coefs

<div dir="rtl">
<b>ملخص الجزء الثاني.</b> تم اختبار أربعة نماذج للانحدار. حقق نموذج Linear Regression (خط الأساس) قيم RMSE = 0.7456 و R² = 0.5758. أظهر نموذج Ridge نتيجة مماثلة تقريباً لأن التنظيم (Regularization) كان ضعيفاً. وكان Lasso الأسوأ (R² = 0.4814) لأن العقوبة كانت مفرطة وألغت معاملات مفيدة.
<br><br>
تمت إضافة متغيرات Polynomial من الدرجة الثانية لـ MedInc و HouseAge. تحسنت النتيجة قليلاً (RMSE = 0.7413 و R² = 0.5806). هذا يدل على وجود علاقة غير خطية لكنها ليست قوية، والعلاقة الخطية كافية.
<br><br>
عند فحص معاملات النموذج، ظهرت AveBedrms كأكبر قيمة (0.783). لكن هذا لا يعني أنها الأهم، بسبب مشكلة التعدد الخطي (Multicollinearity) مع AveRooms، مما جعل المعاملات غير مستقرة. المتغيرات الأهم فعلياً هي MedInc والموقع (Latitude و Longitude). أفضل نموذج كان Polynomial ولكن الفرق بسيط جداً عن الخطي.
</div>

In [ ]:
# logistic
m = df['MedHouseValue'].median()
df['Expensive'] = (df['MedHouseValue'] > m).astype(int)

X_bin = df.drop(columns=['MedHouseValue', 'Expensive'])
y_bin = df['Expensive']

x_tr, x_te, y_tr, y_te = train_test_split(X_bin, y_bin, test_size=0.2, random_state=42)

sc = StandardScaler()
x_tr = sc.fit_transform(x_tr)
x_te = sc.transform(x_te)

log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(x_tr, y_tr)
p_log = log_reg.predict(x_te)

print("Logistic")
print(classification_report(y_te, p_log))
print(confusion_matrix(y_te, p_log))

In [ ]:
# tree (needs unscaled data so split again)
x_tr, x_te, y_tr, y_te = train_test_split(X_bin, y_bin, test_size=0.2, random_state=42)

dt = DecisionTreeClassifier(random_state=42)
dt.fit(x_tr, y_tr)
p_dt = dt.predict(x_te)

print("Tree")
print(classification_report(y_te, p_dt))
print(confusion_matrix(y_te, p_dt))

In [ ]:
# multi class
df['ValueQuartile'] = pd.qcut(df['MedHouseValue'], 4, labels=['Q1', 'Q2', 'Q3', 'Q4'])

X_m = df.drop(columns=['MedHouseValue', 'Expensive', 'ValueQuartile'])
y_m = df['ValueQuartile']

X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(X_m, y_m, test_size=0.2, random_state=42)

rf = RandomForestClassifier(random_state=42)
rf.fit(X_train_m, y_train_m)
p_rf = rf.predict(X_test_m)

print("Multi-class")
print(classification_report(y_test_m, p_rf))
print(confusion_matrix(y_test_m, p_rf))

In [ ]:
# clustering
X_c2 = df.drop(columns=['MedHouseValue', 'Expensive', 'ValueQuartile'])

sc = StandardScaler()
X_sc = sc.fit_transform(X_c2)

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_sc)

print(pca.explained_variance_ratio_)
print(pca.explained_variance_ratio_.sum())

In [ ]:
for k in [3, 4, 5]:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    lb = km.fit_predict(X_sc)
    print("k =", k, "inertia =", round(km.inertia_, 2), "sil =", round(silhouette_score(X_sc, lb), 4))

In [ ]:
km4 = KMeans(n_clusters=4, random_state=42, n_init=10)
lb4 = km4.fit_predict(X_sc)

plt.figure(figsize=(8, 6))
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=lb4, cmap='viridis', s=10, alpha=0.5)
plt.xlabel('PCA 1')
plt.ylabel('PCA 2')
plt.title('KMeans clusters (k=4) in PCA space')
plt.colorbar(label='cluster')
plt.show()

plt.figure(figsize=(8, 6))
plt.scatter(df['Longitude'], df['Latitude'], c=lb4, cmap='viridis', s=10, alpha=0.5)
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.title('KMeans clusters on California map')
plt.colorbar(label='cluster')
plt.show()

In [ ]:
<div dir="rtl">
<b>ملخص الجزء الرابع.</b> تم تطبيق التوحيد (Standardization) ثم PCA. فسر المكونان الأولان 48.85% من التباين. تم استخدام KMeans بقيم k=3,4,5. النتائج:
<br>
- k=3: silhouette = 0.3349
<br>
- k=4: silhouette = 0.3355
<br>
- k=5: silhouette = 0.2931
<br><br>
تنخفض قيمة inertia دائماً مع زيادة k، لذلك لا يُعتمد عليها وحدها. حقق k=4 أفضل قيمة silhouette. يُظهر مخطط PCA والخريطة أن العناقيد متداخلة وبها بعض البنية الجغرافية (ساحلي/داخلي)، لكنها ليست مفصولة بشكل مثالي.
</div>

In [ ]:
<div dir="rtl">
<b>الجزء الخامس: التحليل المقارن</b>
<br><br>
<b>الانحدار:</b> الأفضل كان Polynomial (R² = 0.581) لكن الفرق بسيط عن الخطي. MedInc هو أهم متغير.
<br><br>
<b>التصنيف:</b> الثنائي أفضل دقة (0.84) من المتعدد (0.75). وهذا منطقي لأن المتعدد أصعب.
<br><br>
<b>التجميع:</b> k=4 هو الأفضل (silhouette = 0.3355). وهناك توافق جزئي مع جغرافيا كاليفورنيا.
<br><br>
<b>التعلم المراقب مقابل غير المراقب:</b> التعلم المراقب يوفر نتائج واضحة ومقاييس مثل RMSE و Accuracy. أما غير المراقب فينتج مجموعات يصعب تفسيرها بدقة. كلا الأسلوبين أظهرا أهمية MedInc والموقع.
<br><br>
<b>3 استنتاجات رئيسية:</b>
<br>
1. MedInc هو أهم مؤشر لقيمة المنزل، يليه الموقع (Latitude و Longitude).
<br>
2. إضافة متغيرات Polynomial والتنظيم لم تغير النتيجة كثيراً؛ النموذج الخطي كان كافياً.
<br>
3. التجميع يكشف بنية جغرافية لكنها غير واضحة؛ البيانات بها تداخل (silhouette ≈ 0.33).
</div>

In [ ]:
# hierarchical
a = AgglomerativeClustering(n_clusters=4)
ag_lb = a.fit_predict(X_sc)

print("Hierarchical:", round(silhouette_score(X_sc, ag_lb), 4))
print("KMeans:", round(silhouette_score(X_sc, lb4), 4))

# grid search
p = {
    'n_estimators': [50, 100],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5]
}

g = GridSearchCV(RandomForestClassifier(random_state=42), p, cv=3, n_jobs=-1)
g.fit(X_train_m, y_train_m)

print(g.best_params_)
print("score:", round(g.best_score_, 4))

<div dir="rtl">
<b>ملخص البونص.</b> تم تطبيق التجميع الهرمي (Hierarchical clustering) وكانت نتيجته (0.3232) قريبة من KMeans (0.3355). ووجد GridSearchCV أفضل إعدادات لـ Random Forest: max_depth=None، min_samples_split=5، n_estimators=100، بأفضل دقة = 0.738. وتم إدراج رابط لوحة Streamlit في نهاية الدفتر.
</div>

**STREAMLIT**

[https://aml-f25-all.streamlit.app/](https://aml-f25-all.streamlit.app/)